# Análisis Geoespacial - Barcelona Housing Demographics

**Fecha**: 2026-01-06  
**Autor**: Barcelona Housing Demographics Analyzer  

Este notebook contiene análisis geoespacial y mapas interactivos de Barcelona.

## Contenido
1. [Setup](#1-setup)
2. [Mapas de Precios](#2-precios)
3. [Mapas Demográficos](#3-demografia)
4. [Mapas de Desempleo](#4-desempleo)
5. [Mapas de Turismo](#5-turismo)
6. [Análisis de Clusters](#6-clusters)

## 1. Setup <a id='1-setup'></a>

In [ ]:
# Imports
import sys
from pathlib import Path
import warnings
import json
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from shapely.geometry import shape
import folium
from folium import plugins
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Configuración
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 10)

# Add project root to path
project_root = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.database import DatabaseManager

print("✅ Imports completados")

In [ ]:
# Conectar a la base de datos
db_manager = DatabaseManager()
conn = db_manager.get_connection()

print("✅ Conexión establecida")

In [ ]:
# Cargar geometrías de barrios desde geometry_json
df_barrios_raw = pd.read_sql("""
    SELECT 
        barrio_id,
        barrio_nombre,
        distrito_nombre,
        geometry_json,
        centroide_lat,
        centroide_lon
    FROM dim_barrios
    WHERE geometry_json IS NOT NULL
""", conn)

print(f"📍 Barrios cargados: {len(df_barrios_raw)}")

# Convertir geometry_json a geometrías de Shapely
geometries = []
valid_rows = []

for idx, row in df_barrios_raw.iterrows():
    try:
        if row['geometry_json']:
            geom_dict = json.loads(row['geometry_json'])
            geom = shape(geom_dict)
            geometries.append(geom)
            valid_rows.append(row.drop('geometry_json'))
    except Exception as e:
        print(f"⚠️  Error en barrio {row['barrio_nombre']}: {e}")
        continue

# Crear GeoDataFrame
if geometries:
    gdf_barrios = gpd.GeoDataFrame(
        valid_rows,
        geometry=geometries,
        crs='EPSG:4326'  # WGS84
    )
    
    print(f"✅ Barrios con geometría válida: {len(gdf_barrios)}")
    print(f"📐 CRS: {gdf_barrios.crs}")
    
    # Mostrar mapa base
    fig, ax = plt.subplots(figsize=(12, 12))
    gdf_barrios.plot(ax=ax, edgecolor='black', facecolor='lightblue', alpha=0.5)
    ax.set_title('Mapa de Barrios de Barcelona', fontsize=16, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("❌ No hay geometrías disponibles")
    print("⚠️  Este notebook requiere geometrías para funcionar")
    print("   Solución: Ejecutar script de carga de geometrías")

## 2. Mapas de Precios <a id='2-precios'></a>

In [ ]:
# Cargar precios por barrio (2024)
df_precios_2024 = pd.read_sql("""
    SELECT 
        barrio_id,
        AVG(precio_m2_venta) as precio_medio
    FROM fact_precios
    WHERE anio = 2024 AND precio_m2_venta IS NOT NULL
    GROUP BY barrio_id
""", conn)

# Merge con geometrías
if 'gdf_barrios' in locals():
    gdf_precios = gdf_barrios.merge(df_precios_2024, on='barrio_id', how='left')
    
    print(f"📊 Barrios con datos de precios: {gdf_precios['precio_medio'].notna().sum()}")
    print(f"💰 Precio medio: {gdf_precios['precio_medio'].mean():.2f}€/m²")
else:
    print("⚠️  Sin geometrías disponibles - saltando visualización")

In [ ]:
# Mapa de calor de precios
if 'gdf_precios' in locals() and len(gdf_precios) > 0:
    fig, ax = plt.subplots(1, 1, figsize=(14, 14))
    
    gdf_precios.plot(
        column='precio_medio',
        cmap='RdYlGn_r',
        legend=True,
        edgecolor='black',
        linewidth=0.5,
        ax=ax,
        legend_kwds={'label': 'Precio (€/m²)', 'orientation': 'horizontal', 'shrink': 0.8},
        missing_kwds={'color': 'lightgrey', 'label': 'Sin datos'}
    )
    
    ax.set_title('Mapa de Precios de Vivienda por Barrio (2024)', fontsize=18, fontweight='bold', pad=20)
    ax.axis('off')
    
    # Añadir nombres de barrios más caros
    top_5_caros = gdf_precios.nlargest(5, 'precio_medio')
    for idx, row in top_5_caros.iterrows():
        if row.geometry and not pd.isna(row['precio_medio']):
            centroid = row.geometry.centroid
            ax.annotate(
                row['barrio_nombre'],
                xy=(centroid.x, centroid.y),
                fontsize=8,
                ha='center',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7)
            )
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Sin datos para visualizar")

In [ ]:
# Mapa interactivo con Folium
if 'gdf_precios' in locals() and len(gdf_precios) > 0:
    # Centro de Barcelona
    barcelona_center = [41.3851, 2.1734]
    
    # Crear mapa base
    m = folium.Map(
        location=barcelona_center,
        zoom_start=12,
        tiles='CartoDB positron'
    )
    
    # Añadir capa de coropletas
    folium.Choropleth(
        geo_data=gdf_precios.to_json(),
        name='Precios',
        data=gdf_precios,
        columns=['barrio_id', 'precio_medio'],
        key_on='feature.properties.barrio_id',
        fill_color='YlOrRd',
        fill_opacity=0.7,
        line_opacity=0.2,
        legend_name='Precio Medio (€/m²)',
        nan_fill_color='lightgrey'
    ).add_to(m)
    
    # Añadir tooltips
    folium.GeoJson(
        gdf_precios.to_json(),
        style_function=lambda x: {'fillColor': 'transparent', 'color': 'transparent'},
        tooltip=folium.GeoJsonTooltip(
            fields=['barrio_nombre', 'distrito_nombre', 'precio_medio'],
            aliases=['Barrio:', 'Distrito:', 'Precio (€/m²):'],
            localize=True
        )
    ).add_to(m)
    
    # Guardar mapa
    output_path = project_root / 'notebooks' / 'maps' / 'mapa_precios_2024.html'
    output_path.parent.mkdir(exist_ok=True, parents=True)
    m.save(str(output_path))
    
    print(f"✅ Mapa guardado en: {output_path}")
    display(m)
else:
    print("⚠️  Sin datos para crear mapa interactivo")

## 3. Mapas Demográficos <a id='3-demografia'></a>

In [ ]:
# Cargar datos demográficos
df_demografia = pd.read_sql("""
    SELECT 
        barrio_id,
        poblacion_total,
        densidad_hab_km2
    FROM fact_demografia
    WHERE poblacion_total IS NOT NULL
""", conn)

# Merge con geometrías
if 'gdf_barrios' in locals():
    gdf_demografia = gdf_barrios.merge(df_demografia, on='barrio_id', how='left')
    print(f"📊 Barrios con datos demográficos: {gdf_demografia['poblacion_total'].notna().sum()}")
else:
    print("⚠️  Sin geometrías disponibles")

In [ ]:
# Mapa de densidad poblacional
if 'gdf_demografia' in locals() and len(gdf_demografia) > 0:
    fig, ax = plt.subplots(1, 1, figsize=(14, 14))
    
    gdf_demografia.plot(
        column='poblacion_total',
        cmap='YlOrRd',
        legend=True,
        edgecolor='black',
        linewidth=0.5,
        ax=ax,
        legend_kwds={'label': 'Población', 'orientation': 'horizontal', 'shrink': 0.8},
        missing_kwds={'color': 'lightgrey'}
    )
    
    ax.set_title('Mapa de Población por Barrio', fontsize=18, fontweight='bold', pad=20)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Sin datos para visualizar")

## 4. Mapas de Desempleo <a id='4-desempleo'></a>

In [ ]:
# Cargar datos de desempleo (promedio 2023-2024)
df_desempleo = pd.read_sql("""
    SELECT 
        barrio_id,
        AVG(tasa_desempleo_estimada) as tasa_media
    FROM fact_desempleo
    WHERE tasa_desempleo_estimada IS NOT NULL
    GROUP BY barrio_id
""", conn)

# Merge con geometrías
if 'gdf_barrios' in locals():
    gdf_desempleo = gdf_barrios.merge(df_desempleo, on='barrio_id', how='left')
    print(f"📊 Barrios con datos de desempleo: {gdf_desempleo['tasa_media'].notna().sum()}")
else:
    print("⚠️  Sin geometrías disponibles")

In [ ]:
# Mapa de desempleo
if 'gdf_desempleo' in locals() and len(gdf_desempleo) > 0:
    fig, ax = plt.subplots(1, 1, figsize=(14, 14))
    
    gdf_desempleo.plot(
        column='tasa_media',
        cmap='RdYlGn_r',
        legend=True,
        edgecolor='black',
        linewidth=0.5,
        ax=ax,
        legend_kwds={'label': 'Tasa de Desempleo (%)', 'orientation': 'horizontal', 'shrink': 0.8},
        missing_kwds={'color': 'lightgrey'}
    )
    
    ax.set_title('Mapa de Desempleo por Barrio (2023-2024)', fontsize=18, fontweight='bold', pad=20)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Sin datos para visualizar")

## 5. Mapas de Turismo <a id='5-turismo'></a>

In [ ]:
# Cargar datos de turismo (2024)
df_turismo = pd.read_sql("""
    SELECT 
        barrio_id,
        SUM(num_listings_airbnb) as total_listings
    FROM fact_presion_turistica
    WHERE anio = 2024 AND num_listings_airbnb IS NOT NULL
    GROUP BY barrio_id
""", conn)

# Merge con geometrías
if 'gdf_barrios' in locals():
    gdf_turismo = gdf_barrios.merge(df_turismo, on='barrio_id', how='left')
    gdf_turismo['total_listings'] = gdf_turismo['total_listings'].fillna(0)
    print(f"📊 Barrios con datos de turismo: {(gdf_turismo['total_listings'] > 0).sum()}")
else:
    print("⚠️  Sin geometrías disponibles")

In [ ]:
# Mapa de presión turística
if 'gdf_turismo' in locals() and len(gdf_turismo) > 0:
    fig, ax = plt.subplots(1, 1, figsize=(14, 14))
    
    gdf_turismo.plot(
        column='total_listings',
        cmap='Purples',
        legend=True,
        edgecolor='black',
        linewidth=0.5,
        ax=ax,
        legend_kwds={'label': 'Listings Airbnb', 'orientation': 'horizontal', 'shrink': 0.8}
    )
    
    ax.set_title('Mapa de Presión Turística (Airbnb 2024)', fontsize=18, fontweight='bold', pad=20)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Sin datos para visualizar")

## 6. Análisis de Clusters <a id='6-clusters'></a>

In [ ]:
# Crear dataset para clustering
df_clustering = pd.read_sql("""
    SELECT 
        b.barrio_id,
        b.barrio_nombre,
        AVG(p.precio_m2_venta) as precio,
        AVG(d.poblacion_total) as poblacion,
        AVG(de.tasa_desempleo_estimada) as desempleo,
        SUM(pt.num_listings_airbnb) as turismo
    FROM dim_barrios b
    LEFT JOIN fact_precios p ON b.barrio_id = p.barrio_id AND p.anio = 2024
    LEFT JOIN fact_demografia d ON b.barrio_id = d.barrio_id
    LEFT JOIN fact_desempleo de ON b.barrio_id = de.barrio_id
    LEFT JOIN fact_presion_turistica pt ON b.barrio_id = pt.barrio_id AND pt.anio = 2024
    GROUP BY b.barrio_id, b.barrio_nombre
    HAVING precio IS NOT NULL
""", conn)

# Rellenar NaN solo en columnas numéricas
numeric_cols = ['precio', 'poblacion', 'desempleo', 'turismo']
for col in numeric_cols:
    df_clustering[col] = df_clustering[col].fillna(df_clustering[col].mean())

print(f"📊 Barrios para clustering: {len(df_clustering)}")
df_clustering.head()

In [ ]:
# Normalizar datos
features = ['precio', 'poblacion', 'desempleo', 'turismo']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clustering[features])

# K-Means clustering
n_clusters = 5
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df_clustering['cluster'] = kmeans.fit_predict(X_scaled)

print(f"✅ Clustering completado con {n_clusters} clusters")
print("\nDistribución de barrios por cluster:")
print(df_clustering['cluster'].value_counts().sort_index())

In [ ]:
# Merge con geometrías y visualizar
if 'gdf_barrios' in locals():
    gdf_clusters = gdf_barrios.merge(df_clustering[['barrio_id', 'cluster']], on='barrio_id', how='left')
    
    # Mapa de clusters
    fig, ax = plt.subplots(1, 1, figsize=(14, 14))
    
    gdf_clusters.plot(
        column='cluster',
        cmap='Set3',
        categorical=True,
        legend=True,
        edgecolor='black',
        linewidth=0.8,
        ax=ax,
        legend_kwds={'title': 'Cluster', 'loc': 'upper left'},
        missing_kwds={'color': 'lightgrey'}
    )
    
    ax.set_title('Segmentación de Barrios por Características Socioeconómicas', fontsize=18, fontweight='bold', pad=20)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Sin geometrías - mostrando solo estadísticas")

In [ ]:
# Caracterización de clusters
cluster_stats = df_clustering.groupby('cluster')[features].mean()

print("\n📊 Características de cada Cluster:\n")
print(cluster_stats)

# Visualización de características por cluster
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for idx, feature in enumerate(features):
    ax = axes[idx // 2, idx % 2]
    cluster_stats[feature].plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(f'{feature.capitalize()} por Cluster', fontweight='bold')
    ax.set_xlabel('Cluster')
    ax.set_ylabel(feature.capitalize())
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cerrar conexión
conn.close()
print("✅ Análisis geoespacial completado")
print("✅ Conexión cerrada")